In [ ]:
!pip install -r requirements.txt

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torchvision.models.resnet import BasicBlock
import torch.ao.quantization as quant
import types
import torch.ao.quantization as aq
from torchvision.models.resnet import BasicBlock
from torch.ao.quantization import get_default_qat_qconfig
from torch.ao.quantization.quantize_fx import prepare_qat_fx, convert_fx
# from torch.ao.quantization.pt2e import prepare_qat_pt2e, convert_pt2e
# from torch.ao.quantization.quantizer.qat_config import QATConfig
from torch.ao.quantization.quantizer.xnnpack_quantizer import XNNPACKQuantizer
import copy
from torch.quantization.fake_quantize import FakeQuantizeBase
import brevitas
from tqdm import tqdm

import os
import logging
from datetime import datetime

In [ ]:
os.makedirs("./data", exist_ok=True)
os.makedirs("./logs", exist_ok=True)
os.makedirs("./trained_models", exist_ok=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
transform_train = transforms.Compose([
    transforms.Resize(224),

    # transforms.RandomCrop(32, padding=4),
    # transforms.RandomHorizontalFlip(),
    # transforms.RandomRotation(15),
    # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.Resize(224),

    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform_train)
# Batch_size = 512 and 1024 speed up the process, more than 1024 not
# Currently it takes around 2 minutes epoch for batch_size=256, if need faster, incerasr batch_size
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32,
                                          shuffle=True,
                                          num_workers=1)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=32,
                                         shuffle=False,
                                         num_workers=1)

## Full ResNet18 Traininig

In [ ]:
model = resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

In [ ]:
def training_loop(model, model_name, trainloader, testloader, num_epochs = 10):
    start_of_training_timestamp = datetime.now().strftime("%d.%m.%Y-%H:%M:%S")
    log_filename = f"./logs/{model_name}.log" # _{start_of_training_timestamp}

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_filename),
            logging.StreamHandler()
        ]
    )

    criterion = nn.CrossEntropyLoss()
    params_to_update = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = optim.AdamW(params_to_update, lr=0.001, weight_decay=0.0001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        train_bar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{num_epochs} [Training]")
        for inputs, labels in train_bar:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            # print(outputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            train_bar.set_postfix(
                loss=f"{running_loss / total:.4f}",
                acc=f"{100. * correct / total:.2f}%"
            )
        
        scheduler.step()
        
        train_loss = running_loss / total
        train_acc = 100. * correct / total
        
        model.eval()
        test_loss = 0.0
        correct_test = 0
        total_test = 0

        test_bar = tqdm(testloader, desc=f"Epoch {epoch+1}/{num_epochs} [Validation]")
        with torch.no_grad():
            for inputs, labels in test_bar:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total_test += labels.size(0)
                correct_test += predicted.eq(labels).sum().item()

                test_bar.set_postfix(
                    loss=f"{test_loss / total_test:.4f}",
                    acc=f"{100. * correct_test / total_test:.2f}%"
                )
        
        test_loss /= total_test
        test_acc = 100. * correct_test / total_test
        
        summary_msg = (
            f"Epoch [{epoch+1}/{num_epochs}] -> "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% | "
            f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
        )

        print(summary_msg)
        logging.info(summary_msg)

    return start_of_training_timestamp

In [ ]:
start_of_training_timestamp = training_loop(model, "resnet18_cifar", trainloader, testloader)

In [ ]:
model_path = f"./trained_models/resnet18_cifar10_{start_of_training_timestamp}.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved as {model_path}")

size_bytes = os.path.getsize(model_path)
size_mb = size_bytes / (1024 * 1024)
print(f"Model size: {size_mb:.2f} MB")

## QAT ResNet Training

In [ ]:
qat_model = resnet18()
qat_model.fc = nn.Linear(qat_model.fc.in_features, 10)

if isinstance(trainset, torchvision.datasets.CIFAR10):
    qat_model.conv1 = nn.Conv2d(
        in_channels=3,
        out_channels=64,
        kernel_size=3,
        stride=1,          
        padding=1,          
        bias=False
    )

    # Remove maxpool (not needed for small inputs)
    qat_model.maxpool = nn.Identity()

In [ ]:
print(qat_model)

#### [DEPRECATED] QAT via torch.ao.quantization.prepare_qat (Eager way, older)

In [ ]:
class QuantizableBasicBlock(nn.Module):
    def __init__(self, orig_block: BasicBlock):
        super().__init__()
        # reuse original parameters / submodules
        self.conv1 = orig_block.conv1
        self.bn1 = orig_block.bn1
        self.relu = orig_block.relu
        self.conv2 = orig_block.conv2
        self.bn2 = orig_block.bn2
        self.downsample = orig_block.downsample  # may be None
        self.stride = orig_block.stride

        # local quant/dequant stubs (these use observers when prepared)
        self.quant = aq.QuantStub()
        self.dequant = aq.DeQuantStub()

    def forward(self, x):
        identity = x

        # Quantize at block entry => convs will be fake-quantized during QAT
        out = self.quant(x)

        out = self.conv1(out)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # Dequantize so addition runs in FP32
        out = self.dequant(out)

        if self.downsample is not None:
            # keep downsample in FP32 by applying it directly on FP32 input
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out

In [ ]:
def make_blocks_quantizable(model):
    for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
        layer = getattr(model, layer_name)
        for i in range(len(layer)):
            orig_block = layer[i]
            layer[i] = QuantizableBasicBlock(orig_block)

make_blocks_quantizable(qat_model)

In [ ]:
default_qconfig = get_default_qat_qconfig('fbgemm')
qat_model.qconfig = default_qconfig

qat_model.conv1.qconfig = None
qat_model.fc.qconfig = None

for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
    layer = getattr(qat_model, layer_name)
    for block in layer:
        if getattr(block, 'downsample', None) is not None:
            # set downsample (Sequential) to FP32
            block.downsample.qconfig = None

In [ ]:
qat_model_prepared = torch.ao.quantization.prepare_qat(qat_model)
print(qat_model_prepared)

#### [DEPRECATED] QAT via prepare_qat_fx (New (FX) way. lets you fine-tune per-module quantization in a declarative way)

In [ ]:
from torch.ao.quantization.qconfig import QConfig

class TernaryFakeQuantize(FakeQuantizeBase):
    def __init__(self, threshold=0.05):
        super().__init__()
        self.threshold = threshold  # values near zero → quantize to 0

    def forward(self, X):
        # Quantize weights/activations to {-1, 0, 1}
        X = torch.tanh(X)  # optional normalization step
        out = torch.zeros_like(X)
        out[X > self.threshold] = 1.0
        out[X < -self.threshold] = -1.0
        return out

    def _load_from_state_dict(self, *args, **kwargs):
        # Required to be compatible with torch FX QAT API
        pass
ternary_qconfig = QConfig(
    activation=TernaryFakeQuantize.with_args(threshold=0.05),
    weight=TernaryFakeQuantize.with_args(threshold=0.05)
)

default_qconfig = get_default_qat_qconfig('fbgemm')
qat_model.qconfig = ternary_qconfig # default_qconfig
example_input = torch.randn(1, 3, 32, 32) # For checking 


qconfig_dict = {
    "": ternary_qconfig, # default_qconfig,  # default for all layers
    "module_name": [("conv1", None), ("fc", None)]  # disable quant for first and last
}

qat_model_prepared = prepare_qat_fx(qat_model, qconfig_dict, example_input)
print(qat_model_prepared)

#### QAT via torch.ao.quantization.pt2e (Recommened)

In [ ]:
# https://github.com/Xilinx/brevitas

example_input = (torch.randn(1, 3, 32, 32),)
exported = torch.export.export(qat_model, example_input).module()

quantizer = XNNPACKQuantizer()

In [ ]:
ternary_qat = QATConfig(
    activation_dtype=torch.quint8,  # activations stay 8-bit
    weight_dtype=torch.qint2,       # 2-bit weights → ternary
    enable_observer=True,
    enable_fake_quant=True,
    observer_kwargs={"quant_min": -1, "quant_max": 1}  # enforce ternary range
)

# --- FP32 config for first & last ---
fp32_qat = QATConfig(
    activation_dtype=None,
    weight_dtype=None,
    enable_fake_quant=False,
    enable_observer=False
)

In [ ]:
quantizer.set_global(ternary_qat)

# Disable quantization for first conv and final fc
quantizer.set_module_name("conv1", fp32_qat)
quantizer.set_module_name("fc", fp32_qat)

In [ ]:
qat_prepared = prepare_qat_pt2e(exported, quantizer)

#### QAT Model traininig

In [ ]:
qat_model_prepared.to(device)
start_of_training_timestamp = training_loop(qat_model_prepared, "resnet18_cifar10_qat", trainloader, testloader, num_epochs=1)

In [ ]:
qat_model_prepared.eval()
qat_model_prepared.to("cpu")
# final_quantized_model = convert_fx(qat_model_prepared)
final_quantized_model = convert_pt2e(qat_prepared)

qt_path = f"./trained_models/resnet18_cifar10_qat_{start_of_training_timestamp}.pth"
torch.save(final_quantized_model.state_dict(), qt_path)

print(f"Quantized model saved as {qt_path}")
print("Quantized file size (MB):", os.path.getsize(qt_path)/(1024**2))

In [ ]:
print(final_quantized_model)

#### QAT Model evaluation

In [ ]:
final_quantized_model.load_state_dict(torch.load("./trained_models/resnet18_cifar10_qat_07.10.2025-15:48:04.pth", map_location="cpu"))
final_quantized_model.eval()

In [ ]:
test_loss = 0.0
correct_test = 0
total_test = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to('cpu'), labels.to('cpu')
        outputs = final_quantized_model(inputs)
        criterion = nn.CrossEntropyLoss()
        loss = criterion(outputs, labels)
        test_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total_test += labels.size(0)
        correct_test += predicted.eq(labels).sum().item()

test_loss /= total_test
test_acc = 100. * correct_test / total_test

# Log metrics
print(
    f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
)

In [ ]:
print(final_quantized_model)

## MobileNetV3-Small traininig

In [ ]:
MobileNetV3SmallFull = torchvision.models.mobilenet_v3_small(weights='DEFAULT')
num_features = MobileNetV3SmallFull.classifier[-1].in_features
MobileNetV3SmallFull.classifier[-1] = nn.Linear(num_features, 10)
MobileNetV3SmallFull = MobileNetV3SmallFull.to("cuda")

In [ ]:
start_of_training_timestamp = training_loop(MobileNetV3SmallFull, "MobileNetV3Small_Float32_CIFAR-10", trainloader, testloader, num_epochs = 30)

In [ ]:
MobileNetV3SmallFull.eval()
MobileNetV3SmallFull.to("cpu")

MobileNetV3SmallFull_path = f"./trained_models/MobileNetV3Small_Float32_CIFAR-10_{start_of_training_timestamp}.pth"
torch.save(MobileNetV3SmallFull.state_dict(), MobileNetV3SmallFull_path)

print(f"MobileNetV3Small_Float32_CIFAR-10 saved as {MobileNetV3SmallFull_path}")
print("MobileNetV3Small_Float32_CIFAR-10 file size (MB):", os.path.getsize(MobileNetV3SmallFull_path)/(1024**2))

## MobileNetV3-Small training Brevitas

In [ ]:
from MobileNetV3 import mobilenetv3_small
MobileNetV3SmallFloat32 = mobilenetv3_small()
MobileNetV3SmallFloat32.load_state_dict(torch.load("./trained_models/MobileNetV3Small_FLOAT32_CIFAR-10_UPSCALED.pth"))
MobileNetV3SmallFloat32 = MobileNetV3SmallFloat32.to("cuda")
MobileNetV3SmallFloat32.train()
print()

In [ ]:
from MobileNetV3Brevitas import mobilenetv3_small_quantized
MobileNetV3SmallQuantized = mobilenetv3_small_quantized(weight_bit_width=4, act_bit_width=8)
MobileNetV3SmallQuantized = MobileNetV3SmallQuantized.to("cuda")

In [ ]:
MobileNetV3SmallQuantized.eval()
x = torch.randn(1, 3, 32, 32).to("cuda")  # CIFAR-style input
print("before forward")
out = MobileNetV3SmallQuantized(x)
print("after forward, out.shape:", out.shape)

In [ ]:
MobileNetV3SmallQuantized = mobilenetv3_small_quantized(weight_bit_width=4, act_bit_width=8)
MobileNetV3SmallQuantized.load_state_dict(torch.load("./trained_models/MobileNetV3Small_INT4_CIFAR-10_UPSCALED.pth"))
MobileNetV3SmallQuantized = MobileNetV3SmallQuantized.to("cuda")
MobileNetV3SmallQuantized.train()
print()

In [ ]:
start_of_training_timestamp = training_loop(MobileNetV3SmallQuantized, "MobileNetV3Small_INT4_CIFAR-10_UPSCALED", trainloader, testloader, num_epochs = 200)

In [ ]:
torch.save(MobileNetV3SmallQuantized.state_dict(), "./trained_models/MobileNetV3Small_INT4_CIFAR-10_UPSCALED.pth")

## Model size checks (via Brevitas and QONNX)

In [ ]:
from brevitas.export import export_qonnx

export_path = 'MobileNetV3Small_INT4_QONNX.onnx'

MobileNetV3SmallQuantized.to('cpu')
MobileNetV3SmallQuantized.eval()
export_qonnx(MobileNetV3SmallQuantized, torch.randn(1, 3, 32, 32), export_path=export_path)

In [ ]:
!qonnx-inference-cost MobileNetV3Small_INT2_QONNX.onnx 

In [ ]:
# 5.768096923828125 -> INT32
# 4.028509140014648 -> INT16
# 2.635453224182129 -> INT4

In [ ]:
print(48386304 / (8*1024*1024))

In [ ]:
print(19391492.0 / (8*1024*1024))

In [ ]:
print(22036412.0 / (8*1024*1024))

## Trying regularization on classifier

In [ ]:
for name, param in MobileNetV3SmallQuantized.named_parameters():
    param.requires_grad = False

for name, param in MobileNetV3SmallQuantized.named_parameters():
    if name.startswith('classifier.'):
        param.requires_grad = False

print("--- Parameter requires_grad status ---")
for name, param in MobileNetV3SmallQuantized.named_parameters():
    print(f"{name}: {param.requires_grad}")

In [ ]:
start_of_training_timestamp = training_loop(MobileNetV3SmallQuantized, "MobileNetV3Small_INT4_CIFAR-10_UPSCALED", trainloader, testloader, num_epochs = 10)

## Adverserial attacks check

In [ ]:
import torchattacks

model_dict = {
    # "MobileNetV3_FLOAT32": MobileNetV3SmallFloat32,
    "MobileNetV3_INT4": MobileNetV3SmallQuantized
}

normalization_dict = {'mean': (0.4914, 0.4822, 0.4465), 'std': (0.2023, 0.1994, 0.2010)}

def get_attacks(model):
    eps = 8/255 
    alpha = 2/255
    steps = 10
    
    return {
        "FGSM":   torchattacks.FGSM(model, eps=eps),
        "PGD":    torchattacks.PGD(model, eps=eps, alpha=alpha, steps=steps),
        "MIFGSM": torchattacks.MIFGSM(model, eps=eps, steps=steps, decay=1.0),
        "DIFGSM": torchattacks.DIFGSM(model, eps=eps, alpha=alpha, steps=steps, diversity_prob=0.5, resize_rate=0.9),
        "TIFGSM": torchattacks.TIFGSM(model, eps=eps, alpha=alpha, steps=steps, diversity_prob=0.5),
        "APGD":   torchattacks.APGD(model, eps=eps, steps=steps, norm='Linf')
    }

# --- 3. Benchmarking Loop ---

print(f"{'Model':<20} | {'Attack':<10} | {'Clean Acc':<10} | {'Robust Acc':<10}")
print("-" * 60)

for model_name, model in model_dict.items():
    model = model.to(device)
    model.eval() 
    
    attacks = get_attacks(model)
    
    for attack_name, attack in attacks.items():
        clean_correct = 0
        robust_correct = 0
        total = 0
        
        data_loop = tqdm(testloader, desc=f"Benchmarking {model_name} on {attack_name}", leave=False)
        
        for images, labels in data_loop:
            images, labels = images.to(device), labels.to(device)
            
            with torch.no_grad():
                clean_outputs = model(images)
                _, clean_preds = torch.max(clean_outputs.data, 1)
                clean_correct += (clean_preds == labels).sum().item()
            
            adv_images = attack(images, labels)
            
            with torch.no_grad():
                robust_outputs = model(adv_images)
                _, robust_preds = torch.max(robust_outputs.data, 1)
                robust_correct += (robust_preds == labels).sum().item()
            
            total += labels.size(0)
            
        clean_acc = 100 * clean_correct / total
        robust_acc = 100 * robust_correct / total
        
        print(f"{model_name:<20} | {attack_name:<10} | {clean_acc:.2f}%      | {robust_acc:.2f}%")
    
    print("-" * 60)



## QAT example with Xilinx/Brevitas library

In [ ]:
from brevitas_examples.imagenet_classification.models.mobilenetv1 import MobileNet as MobileNetV1
# from brevitas.quant.scaled_int import Int4WeightPerTensorFloatDecoupled

channels = [[32], [64], [128, 128], [256, 256], [512, 512, 512, 512, 512, 512], [1024, 1024]]
first_stage_stride = False
bit_width = 2
round_avgpool = True

QATMobileNet = MobileNetV1(
    channels=channels,
    first_stage_stride=first_stage_stride,
    round_average_pool=round_avgpool,
    act_bit_width= bit_width,
    weight_bit_width= bit_width,
    last_layer_bit_width= bit_width,
    num_classes=10,
    avg_pool_kernel_size=2,
    first_layer_stride=1,
)

In [ ]:
print(QATMobileNet)

In [ ]:
QATMobileNet.to('cuda')
start_of_training_timestamp = training_loop(QATMobileNet, "MobileNetV1_INT4_CIFAR-10", trainloader, testloader, num_epochs = 1)

In [ ]:
for param in QATMobileNet.parameters():
    param.requires_grad = False

for param in QATMobileNet.output.parameters():
    param.requires_grad = True

for name, param in QATMobileNet.named_parameters():
    if param.requires_grad:
        print(f"Trainable -> {name}")

In [ ]:
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.datatype import DataType
from brevitas.export import export_onnx_qcdq


export_path = 'MobileNetV1_INT2_QONNX.onnx'

QATMobileNet.to('cpu')
QATMobileNet.eval()
export_qonnx(QATMobileNet, torch.randn(1, 3, 32, 32), export_path=export_path)

In [ ]:
qonnx_cleanup(export_path, out_file=export_path)